# CropIQ — Exploratory Data Analysis

Companion to `docs/data_card.md`. Loads the three raw parquets and the assembled
feature matrix; reports row counts, coverage, distributions, and visual sanity
checks. The dashboard and modeling code reuse the same loaders — there is no
ad-hoc cleaning here.

**Sources:**
- `data/raw/nass_yields.parquet` — USDA NASS Quick Stats (Survey, corn grain).
- `data/raw/noaa_weather.parquet` — Open-Meteo Historical Archive (replaces NOAA CAG).
- `data/raw/ssurgo_soil.parquet` — USDA SDA Tabular Service (SSURGO aggregates).
- `data/processed/features.parquet` — built by `src/data/build_dataset.py`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Gotcha #23 — let the notebook import from `src.*` regardless of CWD.
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
%matplotlib inline


## 1. NASS yields

In [ ]:
yields = pd.read_parquet(PROJECT_ROOT / 'data/raw/nass_yields.parquet')
print('shape:', yields.shape)
yields.head()


In [ ]:
yields['state'] = yields['county_fips'].str[:2].map({'19': 'IA', '17': 'IL', '31': 'NE'})
yields.groupby(['state', 'year']).size().unstack().head()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for state, sub in yields.groupby('state'):
    sub.groupby('year')['yield_bu_per_acre'].mean().plot(ax=ax, label=state, marker='o')
ax.set_ylabel('Mean yield (bu/acre)')
ax.set_xlabel('Year')
ax.set_title('Annual state-mean corn grain yield, NASS Survey')
ax.legend(); ax.grid(alpha=0.3); plt.show()


## 2. Weather (Open-Meteo monthly)

In [ ]:
weather = pd.read_parquet(PROJECT_ROOT / 'data/raw/noaa_weather.parquet')
print('shape:', weather.shape)
weather.describe()


In [ ]:
july = weather.loc[weather['month'] == 7].copy()
july['state'] = july['county_fips'].str[:2].map({'19': 'IA', '17': 'IL', '31': 'NE'})
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
july.boxplot(column='tmax_c', by='state', ax=axes[0]); axes[0].set_ylabel('July tmax mean (°C)')
july.boxplot(column='prcp_mm', by='state', ax=axes[1]); axes[1].set_ylabel('July precip total (mm)')
plt.suptitle('July weather by state, 2010–2024'); plt.tight_layout(); plt.show()


## 3. SSURGO soil features

In [ ]:
soil = pd.read_parquet(PROJECT_ROOT / 'data/raw/ssurgo_soil.parquet')
print('shape:', soil.shape)
soil.describe()


## 4. Feature matrix

In [ ]:
from src.concepts.feature_matrix import load as load_features, REQUIRED_COLUMNS
features = load_features(PROJECT_ROOT / 'data/processed/features.parquet')
print('shape:', features.shape)
for c in REQUIRED_COLUMNS:
    print(f'  {c:<35} non-null {features[c].notna().mean():.1%}')


In [ ]:
features.corr(numeric_only=True)['yield_bu_per_acre'].drop('yield_bu_per_acre').sort_values()
